# Extracción y tracking de endpoints de portales MP
Este notebook extrae, procesa y guarda datos de varios endpoints del Ministerio Público, generando datasets, tracking de fechas y atributos dinámicos, y un datapackage.json por carpeta.

In [20]:
# 1. Importar librerías necesarias
import requests
import json
from datetime import datetime
import os

In [ ]:
# 2. Definir endpoints y rutas de almacenamiento
datasets = [
    {
        "name": "endpoint_perdida-dominio-bienes",
        "title": "Pérdida de Dominio de Bienes",
        "endpoint": "https://justicialibre-gw.mp.gob.bo/v1/portal-web/perdida-dominio-bienes"
    },
    {
        "name": "endpoint_personas-rebeldia",
        "title": "Personas en Rebeldía",
        "endpoint": "https://justicialibre-gw.mp.gob.bo/v1/portal-web/personas-rebeldia"
    },
    {
        "name": "endpoint_edictos",
        "title": "Edictos",
        "endpoint": "https://justicialibre-gw.mp.gob.bo/v1/portal-web/edictos"
    },
    {
        "name": "endpoint_personas-desaparecidas",
        "title": "Personas Desaparecidas",
        "endpoint": "https://justicialibre-gw.mp.gob.bo/v1/portal-web/personas-desaparecidas"
    }
]
base_dir = '../data/'

In [22]:
# 3. Función para aplanar diccionarios anidados
def flatten(d, parent_key='', sep='_'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

In [ ]:
# 4. Función para extraer, procesar y guardar datos de un endpoint (paginación)
def process_endpoint(dataset):
    name = dataset['name']
    title = dataset['title']
    endpoint = dataset['endpoint']
    folder = os.path.join(base_dir, name)
    os.makedirs(folder, exist_ok=True)
    json_path = os.path.join(folder, f'{name}.json')
    raw_path = os.path.join(folder, f'{name}_raw.json')
    result = []
    all_raw = []
    page = 1
    size = 100
    total = None
    now = datetime.now().isoformat()
    # Cargar dataset previo si existe
    if os.path.exists(json_path):
        with open(json_path, 'r', encoding='utf-8') as f:
            prev = {str(row.get('id', i)): row for i, row in enumerate(json.load(f))}
    else:
        prev = {}
    while True:
        payload = {"size": size, "page": page, "where": {}}
        headers = {"Content-Type": "application/json"}
        response = requests.post(endpoint, json=payload, headers=headers)
        data = response.json()
        all_raw.append(data)
        if not data.get('response') or not data['response'].get('data'):
            break
        for i, item in enumerate(data['response']['data']):
            row_id = str(item.get('id', f"{page}_{i}"))
            flat = flatten(item)
            if row_id in prev:
                primera_vez = prev[row_id].get('primera_vez_visto', now)
            else:
                primera_vez = now
            ultima_vez = now
            flat['primera_vez_visto'] = primera_vez
            flat['ultima_vez_visto'] = ultima_vez
            result.append(flat)
        # Paginación
        pagination = data['response'].get('pagination', {})
        total = pagination.get('total', None)
        current = (page-1) * size + min(size, len(data['response']['data']))
        print(f"{title=} {page=} {current=} {total=}")
        if total is None or current >= total:
            break
        page += 1
    # Guardar la data en bruto para auditoría
    with open(raw_path, 'a', encoding='utf-8') as f:
        for raw in all_raw:
            f.write(json.dumps(raw, ensure_ascii=False) + '\n')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f'Dataset actualizado: {json_path}')

In [ ]:
# 5. Procesar y guardar datos para cada endpoint
for dataset in datasets:
    process_endpoint(dataset)

title='Pérdida de Dominio de Bienes' page=1 total=814 total=814
title='Pérdida de Dominio de Bienes' page=2 total=814 total=814
title='Pérdida de Dominio de Bienes' page=3 total=814 total=814
title='Pérdida de Dominio de Bienes' page=4 total=814 total=814
title='Pérdida de Dominio de Bienes' page=5 total=814 total=814
title='Pérdida de Dominio de Bienes' page=6 total=814 total=814
title='Pérdida de Dominio de Bienes' page=7 total=814 total=814
title='Pérdida de Dominio de Bienes' page=8 total=814 total=814
title='Pérdida de Dominio de Bienes' page=9 total=814 total=814
Dataset actualizado: ../data/datasets/perdida-dominio-bienes/perdida-dominio-bienes.json
title='Personas en Rebeldía' page=1 total=29497 total=29497
title='Personas en Rebeldía' page=2 total=29497 total=29497
title='Personas en Rebeldía' page=3 total=29497 total=29497
title='Personas en Rebeldía' page=4 total=29497 total=29497
title='Personas en Rebeldía' page=5 total=29497 total=29497
title='Personas en Rebeldía' page=6

In [ ]:
# 6. Crear archivos datapackage.json para cada dataset
def create_datapackage(dataset):
    name = dataset['name']
    title = dataset['title']
    folder = os.path.join(base_dir, name)
    datapackage = {
        "profile": "tabular-data-package",
        "name": name,
        "title": title,
        "description": f"Dataset extraído del endpoint {dataset['endpoint']}",
        "resources": [
            {
                "path": f"{name}.json",
                "name": name,
                "profile": "tabular-data-resource",
                "format": "json",
                "mediatype": "application/json"
            }
        ]
    }
    with open(os.path.join(folder, 'datapackage.json'), 'w', encoding='utf-8') as f:
        json.dump(datapackage, f, ensure_ascii=False, indent=2)
    print(f'Datapackage creado: {os.path.join(folder, "datapackage.json")}')

for dataset in datasets:
    create_datapackage(dataset)

Datapackage creado: ../data/datasets/perdida-dominio-bienes/datapackage.json
Datapackage creado: ../data/datasets/personas-rebeldia/datapackage.json
Datapackage creado: ../data/datasets/edictos/datapackage.json
Datapackage creado: ../data/datasets/personas-desaparecidas/datapackage.json
